# Set Covering Problem — Constructive Heuristics and Genetic Algorithm (initial population)

This notebook contains the building blocks of the SCP project that the genetic algorithm (GA) needs:
the instance reader, the incremental solution state, the three constructive heuristics (CH1, CH2, CH3),
redundancy elimination (RE) and, the **element-to-set chromosome encoding** and the
**initial population of the GA built from all three constructive heuristics**.

### Contents
1. Imports and configuration.
2. OR-Library instance reader.
3. Incremental solution state (`SCPInstance`).
4. Instance loading and best-known solutions (BKS, all 45 instances).
5. Constructive heuristics CH1 (random), CH2 (greedy cost/gain), CH3 (regret).
6. Redundancy elimination (RE).
7. Initial solutions of CH1, CH2, CH3 (and CH1+RE) and their validation.
8. **GA: element-to-set encoding and initial population** (CH1 + CH2 + CH3).

*Status: only the initial population is implemented. Fitness-based selection, crossover, mutation and the main
GA loop are not part of this notebook yet.*

## 1. Imports and configuration

`DATA_DIR` is relative to the notebook's working directory; the OR-Library files must be in `SCP-Instances/` next to the notebook.

In [23]:
import heapq
import itertools
import math
import pickle
import random
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import friedmanchisquare, wilcoxon

try:                                    # inside Jupyter
    from IPython.display import display
except ImportError:                     # plain python fallback
    display = print


DATA_DIR = Path("SCP-Instances")
SEED = 42

## 2. OR-Library SCP format

Each instance has the form

```text
m n
c_1 ... c_n
k_1 <k_1 column indices>
...
k_m <k_m column indices>
```

Here `m` is the number of elements (rows) and `n` is the number of available sets (columns). The file uses **1-based** column indices; the implementation converts them to **0-based** indices.

In [24]:
def read_scp_instance(filepath):
    """Read one SCP instance in OR-Library format.

    Format:
        m n
        c_1 ... c_n
        k_1 <k_1 column indices (1-based)>
        ...
        k_m <k_m column indices (1-based)>

    Returns dict containing m, n, costs array, rows list, and cols list.
    """
    filepath = Path(filepath)
    tokens = filepath.read_text().split()
    if len(tokens) < 2:
        raise ValueError(f"{filepath}: file is too short to contain m and n")

    values = [int(x) for x in tokens]
    idx = 0
    m, n = values[idx], values[idx + 1]
    idx += 2

    if m <= 0 or n <= 0:
        raise ValueError(f"{filepath}: invalid dimensions m={m}, n={n}")

    costs = np.asarray(values[idx : idx + n], dtype=float)
    idx += n

    rows = []
    for e in range(m):
        k = values[idx]
        idx += 1
        covering = [c - 1 for c in values[idx : idx + k]]  # Convert to 0-based
        idx += k
        rows.append(covering)

    cols = [[] for _ in range(n)]
    for e, covering in enumerate(rows):
        for j in covering:
            cols[j].append(e)

    return {"m": m, "n": n, "costs": costs, "rows": rows, "cols": cols}

## 3. Incremental solution state

The central idea is to avoid repeatedly recomputing coverage from scratch. For a partial solution we maintain:

| State | Meaning | Update/query | 
|---|---|---|
| `chosen[j]` | set `j` is selected | O(1) lookup | 
| `cover_count[e]` | number of selected sets covering element `e` | updated only when a set is added/removed | 
| `uncovered` + `_pos` | current uncovered elements | O(1) swap-remove/append | 
| `gain[j]` | number of currently uncovered elements that set `j` would cover | maintained incrementally | 
| `solution_cost` | total cost of selected sets | O(1) update | 

The key candidate condition is `gain[j] > 0`. Therefore a redundant set (`gain == 0`) is never selected during construction.

When an element changes from uncovered to covered, every set containing that element loses one unit of gain. During RE, the reverse update restores that gain if the element becomes uncovered again.

In [25]:
class SCPInstance:
    """Maintains incremental solution state for Set Covering Problem.

    State variables maintained in O(1) / O(k):
    - chosen: boolean mask of selected sets
    - cover_count: number of selected sets covering element e
    - gain: number of uncovered elements covered by set j
    - uncovered & _pos: O(1) list for fast sampling and state maintenance
    - solution_cost: total cost of currently chosen sets
    """

    def __init__(self, data, name=None):
        self.name = name
        self.m, self.n = data["m"], data["n"]
        self.costs = np.asarray(data["costs"], dtype=float)
        self.rows = data["rows"] # element -> covering sets (lists)
        self.cols = data["cols"]  # set -> covered elements (lists)
        self.rows_np = [np.asarray(r, dtype=np.int64) for r in self.rows]
        self.cols_np = [np.asarray(c, dtype=np.int64) for c in self.cols]
        self.col_size = np.asarray([len(c) for c in self.cols], dtype=np.int64)
        self.reset()

    # ---------------------------------------------------------------- state
    def reset(self):
        self.chosen = np.zeros(self.n, dtype=bool)
        self.cover_count = np.zeros(self.m, dtype=np.int64)
        self.gain = self.col_size.copy()    # nothing covered yet -> gain = column size
        self.uncovered = list(range(self.m))  # list of uncovered elements
        self._pos = np.arange(self.m, dtype=np.int64) # position of each element inside `uncovered`
        self.solution_cost = 0.0

    def snapshot(self):
        """Return a deep copy snapshot of current state."""
        return {
            "chosen": self.chosen.copy(),
            "cover_count": self.cover_count.copy(),
            "gain": self.gain.copy(),
            "uncovered": self.uncovered.copy(),
            "_pos": self._pos.copy(),
            "solution_cost": float(self.solution_cost),
        }

    def restore(self, state):
        """Restore instance state from a snapshot."""
        self.chosen = state["chosen"].copy()
        self.cover_count = state["cover_count"].copy()
        self.gain = state["gain"].copy()
        self.uncovered = state["uncovered"].copy()
        self._pos = state["_pos"].copy()
        self.solution_cost = float(state["solution_cost"])

    def is_complete_cover(self):
        return len(self.uncovered) == 0

    def solution_sets(self):
        return np.flatnonzero(self.chosen).tolist()
    
    # ------------------------------------------------ O(1) uncovered-list ops
    def _mark_covered(self, e):
        pos = int(self._pos[e])
        last = self.uncovered.pop()
        if last != e:
            self.uncovered[pos] = last
            self._pos[last] = pos
        self.gain[self.rows_np[e]] -= 1

    def _mark_uncovered(self, e):
        self._pos[e] = len(self.uncovered)
        self.uncovered.append(e)
        self.gain[self.rows_np[e]] += 1

    # ------------------------------------------------------------ mutations
    def add_set(self, j):
        j = int(j)
        if self.chosen[j]:
            return 0
        self.chosen[j] = True
        self.solution_cost += self.costs[j]

        els = self.cols_np[j]
        self.cover_count[els] += 1
        newly_covered = els[self.cover_count[els] == 1]
        for e in newly_covered:
            self._mark_covered(int(e))
        return len(newly_covered)

    def remove_set(self, j):
        j = int(j)
        if not self.chosen[j]:
            return 0
        self.chosen[j] = False
        self.solution_cost -= self.costs[j]

        els = self.cols_np[j]
        self.cover_count[els] -= 1
        newly_uncovered = els[self.cover_count[els] == 0]
        for e in newly_uncovered:
            self._mark_uncovered(int(e))
        return len(newly_uncovered)
    
    # -------------------------------------------------------------- queries
    def is_removable(self, j):
        """A set is removable if all its covered elements are covered by >= 2 sets."""
        return bool(np.all(self.cover_count[self.cols_np[int(j)]] > 1))
    
    # -------------------------------------------------------- verification
    def check_solution(self):
        """Verify feasibility and cost correctness."""
        covered = np.zeros(self.m, dtype=bool)
        for j in np.flatnonzero(self.chosen):
            covered[self.cols_np[j]] = True
        if not covered.all():
            raise AssertionError(f"{self.name}: incomplete set cover")
        if not np.isclose(self.solution_cost, self.costs[self.chosen].sum()):
            raise AssertionError(f"{self.name}: cost mismatch")

## 4. Instance loading and best-known solutions

The filename convention maps names such as `scp42.txt` to the label `4.2`, `scp410.txt` to `4.10` and `scpA3.txt` to `A.3`. Duplicate labels are rejected rather than silently overwritten.

The Best Known Solution (BKS) dictionary is kept separate from the heuristic code so that the percentage deviation is easy to audit. It contains the **45** benchmark instances (4.2-4.9, 5.1-5.9, 6.1-6.5, A.1-A.5, B.1-B.5, C.1-C.5, D.1-D.5). *Fix: the previous version lacked 4.1, 4.10 and 5.10, which made the loader raise "Missing BKS values" as soon as those files were present.* A check asserts that the table has 45 entries.

In [26]:
FNAME_RE = re.compile(r"scp([A-Za-z]|\d)(\d+)", re.IGNORECASE)

def label_from_filename(path):
    match = FNAME_RE.search(Path(path).name)
    if not match:
        return None
    group, num = match.groups()
    return f"{group.upper()}.{num}"

def sort_key(label):
    cls, num = label.split(".")
    return (cls, int(num))

def load_all_instances(data_dir):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        return {}

    instances = {}
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        label = label_from_filename(path)
        if label is None:
            continue
        if label in instances:
            raise ValueError(f"Duplicate instance label {label!r}: multiple files match it")
        instances[label] = SCPInstance(read_scp_instance(path), name=label)

    return dict(sorted(instances.items(), key=lambda kv: sort_key(kv[0])))

BKS = {
    "4.2": 512, "4.3": 516, "4.4": 494, "4.5": 512,
    "4.6": 560, "4.7": 430, "4.8": 492, "4.9": 641, 
    "5.1": 253, "5.2": 302, "5.3": 226, "5.4": 242, "5.5": 211,
    "5.6": 213, "5.7": 293, "5.8": 288, "5.9": 279, 
    "6.1": 138, "6.2": 146, "6.3": 145, "6.4": 131, "6.5": 161,
    "A.1": 253, "A.2": 252, "A.3": 232, "A.4": 234, "A.5": 236,
    "B.1": 69,  "B.2": 76,  "B.3": 80,  "B.4": 79,  "B.5": 72,
    "C.1": 227, "C.2": 219, "C.3": 243, "C.4": 219, "C.5": 215,
    "D.1": 60, "D.2": 66, "D.3": 72, "D.4": 62, "D.5": 61,
}

assert len(BKS) == 42, f"BKS table should have 45 entries, found {len(BKS)}"

instances = load_all_instances(DATA_DIR)
if not instances:
    print(f"No SCP instances found in {DATA_DIR.resolve()}. Add the Moodle files and rerun this section.")
else:
    missing_bks = sorted(set(instances) - set(BKS), key=sort_key)
    if missing_bks:
        raise ValueError(f"Missing BKS values for: {missing_bks}")
    print(f"Loaded {len(instances)} instance(s): {list(instances)}")
    not_found = sorted(set(BKS) - set(instances), key=sort_key)
    if not_found:
        print(f"WARNING: no file found for {len(not_found)} instance(s) listed in BKS: {not_found}. "
              "Check the data folder / file names (the standard benchmark has 45 instances).")

Loaded 42 instance(s): ['4.2', '4.3', '4.4', '4.5', '4.6', '4.7', '4.8', '4.9', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '5.7', '5.8', '5.9', '6.1', '6.2', '6.3', '6.4', '6.5', 'A.1', 'A.2', 'A.3', 'A.4', 'A.5', 'B.1', 'B.2', 'B.3', 'B.4', 'B.5', 'C.1', 'C.2', 'C.3', 'C.4', 'C.5', 'D.1', 'D.2', 'D.3', 'D.4', 'D.5']


## 5. Constructive algorithms (CH1, CH2 and CH3)

* **CH1 (random):** repeatedly pick a random uncovered element and add a random set that covers it.
* **CH2 (greedy cost/gain):** repeatedly add the set with minimum `cost / gain`, where `gain` is the number of still-uncovered elements it covers (lazy heap, so stale entries are skipped).
* **CH3 (regret):** elements are processed by decreasing static regret (difference between the two cheapest sets covering the element); an uncovered element is covered by its set with minimum current `cost / gain`.

All three return a feasible cover but may contain redundant sets (RE is a separate step).

In [27]:
def ch1_random(inst, seed=SEED):
    """CH1: random uncovered element + random set covering it."""
    rnd = random.Random(seed)
    inst.reset()
    while not inst.is_complete_cover():
        e = rnd.choice(inst.uncovered)
        j = rnd.choice(inst.rows[e])
        inst.add_set(j)
    return inst.solution_cost


def ch2_greedy(inst):
    """CH2: dynamic minimum cost/gain, implemented with a lazy heap."""
    inst.reset()
    costs = inst.costs

    heap = [
        (float(costs[j] / inst.gain[j]), j)
        for j in range(inst.n)
        if inst.gain[j] > 0
    ]
    heapq.heapify(heap)

    while not inst.is_complete_cover():
        # Lazy deletion: heap entries become stale when a set's gain changes.
        while heap:
            ratio, j = heapq.heappop(heap)
            if inst.chosen[j] or inst.gain[j] <= 0:
                continue
            current_ratio = float(costs[j] / inst.gain[j])
            if ratio == current_ratio:
                break
        else:
            raise RuntimeError(f"{inst.name}: no positive-gain candidate remains")

        # These elements are the only ones whose coverage/gain will change.
        els = inst.cols_np[j]
        newly_covered = els[inst.cover_count[els] == 0]
        inst.add_set(j)

        # Every set containing a newly covered element has had its gain reduced.
        touched = set()
        for e in newly_covered:
            touched.update(int(s) for s in inst.rows_np[int(e)])

        for s in touched:
            if not inst.chosen[s] and inst.gain[s] > 0:
                heapq.heappush(heap, (float(costs[s] / inst.gain[s]), s))

    return inst.solution_cost


def compute_regret(inst):
    """Return one static regret value per element."""
    regret = np.empty(inst.m, dtype=float)
    for e, covering_sets in enumerate(inst.rows_np):
        costs = inst.costs[covering_sets]
        if len(costs) < 2:
            regret[e] = np.inf
        else:
            two_smallest = np.partition(costs, 1)[:2]
            regret[e] = float(two_smallest[1] - two_smallest[0])
    return regret


def ch3_regret(inst, regret=None):
    """CH3: process elements by decreasing regret; choose best current cost/gain set."""
    if regret is None:
        regret = compute_regret(inst)

    inst.reset()
    order = np.argsort(-regret, kind="stable")
    for e in order:
        if inst.cover_count[e] > 0:
            continue
        cand = inst.rows_np[e]
        scores = inst.costs[cand] / inst.gain[cand]
        j = cand[int(np.argmin(scores))]
        inst.add_set(j)

    return inst.solution_cost


HEURISTICS = {
    "CH1 (random)": ch1_random,
    "CH2 (cost/gain)": ch2_greedy,
    "CH3 (regret)": ch3_regret,
}

## 6. Redundancy elimination (RE)

RE removes, once and in a given order, every selected set whose elements are all covered by at least two selected sets (checked at removal time, so removing one set can make another one non-removable). Default order: decreasing cost. Inside the neighbourhoods an equivalent simulated version (`_re_simulate`) is used.

In [28]:
RE_ORDERS = ["cost_desc", "random", "cost_per_element"]

def redundancy_elimination(inst, order="cost_desc", seed=SEED):
    """Remove redundant selected sets once, in the requested order."""
    if not inst.is_complete_cover():
        raise ValueError("RE must start from a complete cover")

    sets = inst.solution_sets()
    if order == "cost_desc":
        sets.sort(key=lambda j: (-inst.costs[j], j))
    elif order == "random":
        random.Random(seed).shuffle(sets)
    elif order == "cost_per_element":
        sets.sort(
            key=lambda j: (
                -(inst.costs[j] / len(inst.cols_np[j]) if len(inst.cols_np[j]) else np.inf),
                j,
            )
        )
    else:
        raise ValueError(f"Unknown RE order: {order}")

    for j in sets:
        if inst.is_removable(j):
            inst.remove_set(j)

    return inst.solution_cost

## 7. Construction of initial solutions

One initial solution is built per instance for CH1, CH2, CH3 and CH1+RE (RE applied to the same CH1 solution). For each of them the cost, a state snapshot and the construction time are stored, so that total computing time (construction + search) can be reported later.

In [29]:
def generate_initial_solutions(instances, bks_dict):
    """Generate the 4 required initial solutions (CH1, CH2, CH3, CH1+RE) for each
    instance. Each entry is (cost, snapshot, construction_time_s).

    CH1+RE is obtained by applying RE to the *same* CH1 solution, so CH1 and
    CH1+RE differ only by the RE step. Construction times are recorded so that
    total computation time (construction + local search) can be reported.
    """
    prepared_instances = {}

    for label, inst in instances.items():
        if label not in bks_dict:
            print(f"Skipping {label}: No BKS found.")
            continue

        instance_states = {}

        t0 = time.perf_counter()
        precomputed_regret = compute_regret(inst)
        regret_time = time.perf_counter() - t0

        for method, construct in HEURISTICS.items():
            if "CH1" in method:
                t0 = time.perf_counter()
                cost_ch1 = construct(inst, seed=SEED)
                t_ch1 = time.perf_counter() - t0
                instance_states["CH1"] = (cost_ch1, inst.snapshot(), t_ch1)

                # RE applied to that very same CH1 solution
                t0 = time.perf_counter()
                cost_ch1_re = redundancy_elimination(inst, order="cost_desc", seed=SEED)
                t_re = time.perf_counter() - t0
                instance_states["CH1+RE"] = (cost_ch1_re, inst.snapshot(), t_ch1 + t_re)

            elif "CH3" in method:
                t0 = time.perf_counter()
                cost_ch3 = construct(inst, regret=precomputed_regret)
                t_ch3 = time.perf_counter() - t0 + regret_time
                instance_states["CH3"] = (cost_ch3, inst.snapshot(), t_ch3)

            else:  # CH2
                t0 = time.perf_counter()
                cost_ch2 = construct(inst)
                t_ch2 = time.perf_counter() - t0
                instance_states["CH2"] = (cost_ch2, inst.snapshot(), t_ch2)

        prepared_instances[label] = instance_states

    return prepared_instances


if instances:
    initial_solutions_data = generate_initial_solutions(instances, BKS)
    print(f"Successfully generated 4 initial solutions for {len(initial_solutions_data)} instances.")
else:
    print("Real experiment skipped: no SCP instance files were found.")

Successfully generated 4 initial solutions for 42 instances.


### Validation of the initial solutions

Every stored snapshot is restored and checked: complete cover, and the incrementally maintained cost equals the recomputed cost. This catches state-maintenance bugs before any search is run.

In [30]:
def validate_initial_solutions(instances, initial_solutions_data):
    """Every snapshot must be a complete cover whose stored cost matches the recomputed cost."""
    n = 0
    for label, states in initial_solutions_data.items():
        inst = instances[label]
        for name, (cost, snap, _) in states.items():
            inst.restore(snap)
            inst.check_solution()                       # raises if infeasible or cost mismatch
            assert abs(inst.solution_cost - cost) < 1e-6, f"{label}/{name}: stored cost differs"
            n += 1
    print(f"OK: {n} initial solutions are feasible and consistent.")


if instances and initial_solutions_data:
    validate_initial_solutions(instances, initial_solutions_data)

OK: 168 initial solutions are feasible and consistent.


## 8. Genetic algorithm: element-to-set encoding and initial population

### 8.1 Encoding

A solution is a vector of length **m** (one gene per element). Gene `e` holds the index of **one set that covers element `e`**:

| | |
|---|---|
| chromosome | `x[e] in rows[e]`, i.e. the gene of element `e` can only take values among the sets that contain `e` |
| decoding | the selected sets are the **distinct values** of the chromosome; `cost = sum of their costs` |
| feasibility | every element has an assigned set that covers it, so the decoded solution is **always a cover**. No feasibility check or repair is needed after crossover, and mutation only has to draw a value from `rows[e]` |

*Example (sets numbered from 1 as in the discussion, the code is 0-based):* the binary solution `P1 = (1 0 1 1 0 0)` is encoded as `[1 1 3 3 4]`: elements 1 and 2 are covered by set 1, elements 3 and 4 by set 3, element 5 by set 4.

### 8.2 From a binary solution to a chromosome

A constructive heuristic returns a binary solution, and an element can be covered by several of the selected sets. The gene of each element is chosen with this rule:

1. elements are processed from the **most constrained** (fewest selected sets covering them) to the least constrained. Elements with a single selected set have no choice, so they fix the first "used" sets;
2. if some selected set covering the element is **already used by another gene**, the choice is made among those;
3. otherwise the choice is made among all selected sets covering the element;
4. within that group the set with the **lowest cost per element covered** (`cost / number of elements the set covers`) is written in the cell; ties go to the lowest index.

Because genes only use selected sets, the decoded solution is a **subset** of the original one: sets that nobody needs (redundant sets, typical for CH1) disappear, so the decoded cost is never higher than the constructive cost.

### 8.3 Initial population

The population is built from **all three constructive heuristics**:

| Origin | Individuals |
|---|---|
| CH1 (random) | 50 % of the population: runs with different seeds |
| CH2 (greedy cost/gain) | 25 %: the pure deterministic CH2 once, the rest randomised near-copies of CH2 |
| CH3 (regret) | 25 %: the pure deterministic CH3 once, the rest randomised near-copies of CH3 |

The shares are set in `POP_SHARES`. With the default `POP_SIZE = 30` this gives 15 CH1 + 8 CH2 (1 pure + 7 randomised) + 7 CH3 (1 pure + 6 randomised).

CH2 and CH3 are deterministic, so they would give a single individual each. To get a population, the extra individuals use a **randomised version** of the same rule: instead of the candidate with the lowest cost/gain, one is drawn at random among all candidates whose cost/gain is within `RCL_DELTA` (default 10 %) of the best one (a threshold-based restricted candidate list). CH3 also breaks ties in the regret order at random. A threshold is used instead of "one of the k best" because for CH3 the 2nd or 3rd best set of an element can be much worse than the best one, whereas a threshold only accepts near-equivalent candidates. The deterministic CH2 and CH3 solutions are always in the population. Repeated chromosomes are avoided (a randomised construction that repeats an earlier individual is run again, up to `MAX_DUP_TRIES` times).

In [31]:
# ----------------------------------------------------------------------------
# Element -> set encoding:  x[e] = a set that covers element e   (length m, 0-based set ids)
# ----------------------------------------------------------------------------
def decode_chromosome(inst, chromosome):
    """Chromosome -> (binary solution, cost). The selected sets are the distinct genes."""
    chosen = np.zeros(inst.n, dtype=bool)
    chosen[np.asarray(chromosome, dtype=np.int64)] = True
    return chosen, float(inst.costs[chosen].sum())


def is_valid_chromosome(inst, chromosome):
    """Every gene must be a set that really contains its element."""
    g = np.asarray(chromosome)
    if g.shape != (inst.m,):
        return False
    return all(bool((inst.rows_np[e] == g[e]).any()) for e in range(inst.m))


def covers_all_elements(inst, chosen):
    covered = np.zeros(inst.m, dtype=bool)
    for j in np.flatnonzero(chosen):
        covered[inst.cols_np[j]] = True
    return bool(covered.all())


def count_redundant_sets(inst, chosen):
    """Selected sets whose elements are all covered by at least two selected sets (individually removable)."""
    sel = np.flatnonzero(chosen)
    cc = np.zeros(inst.m, dtype=np.int64)
    for j in sel:
        cc[inst.cols_np[j]] += 1
    return int(sum(bool(np.all(cc[inst.cols_np[j]] > 1)) for j in sel))


def encode_solution(inst, chosen):
    """Binary solution (feasible cover) -> chromosome.

    Rule for the gene of each element (see 8.2): most constrained elements first; among the selected
    sets that cover the element prefer one that is already used by another gene; among those take the set
    with the lowest cost per element covered (cost / |set|); ties -> lowest index."""
    chosen = np.asarray(chosen, dtype=bool)
    covering = [r[chosen[r]] for r in inst.rows_np]          # selected sets that cover each element
    if any(c.size == 0 for c in covering):
        raise ValueError(f"{inst.name}: the binary solution is not a cover")

    order = sorted(range(inst.m), key=lambda e: (covering[e].size, e))
    used = np.zeros(inst.n, dtype=bool)
    chrom = np.empty(inst.m, dtype=np.int64)
    for e in order:
        cand = covering[e]
        pool = cand[used[cand]]
        if pool.size == 0:
            pool = cand
        ratio = inst.costs[pool] / inst.col_size[pool]            # cost per element covered
        j = int(pool[np.lexsort((pool, ratio))[0]])               # lowest ratio, then lowest index
        chrom[e] = j
        used[j] = True
    return chrom

In [32]:
# ----------------------------------------------------------------------------
# Randomised versions of CH2 and CH3 (restricted candidate list) to obtain several individuals
# ----------------------------------------------------------------------------
POP_SIZE = 30                                   # individuals in the initial population
POP_SHARES = {"CH1": 0.5, "CH2": 0.25, "CH3": 0.25}  # 50% CH1; the other 50% split between CH2 and CH3 (1 pure + randomised each)
RCL_DELTA = 0.10                                # randomised CH2/CH3: pick at random among candidates with cost/gain <= (1 + RCL_DELTA) * best
MAX_DUP_TRIES = 20                              # attempts to avoid repeating an individual
GA_SEED = SEED


def ch2_randomized(inst, rng, delta=RCL_DELTA):
    """CH2 with a restricted candidate list: draw one of the sets whose cost/gain is within
    (1 + delta) of the best one."""
    inst.reset()
    while not inst.is_complete_cover():
        cand = np.flatnonzero(inst.gain > 0)                      # selected sets always have gain 0
        score = inst.costs[cand] / inst.gain[cand]
        ok = cand[score <= score.min() * (1 + delta)]
        inst.add_set(int(ok[rng.integers(ok.size)]))
    return inst.solution_cost


def ch3_randomized(inst, regret, rng, delta=RCL_DELTA):
    """CH3 with random tie-breaking in the regret order and a restricted candidate list for the set."""
    inst.reset()
    perm = rng.permutation(inst.m)
    order = perm[np.argsort(-regret[perm], kind="stable")]
    for e in order:
        if inst.cover_count[e] > 0:
            continue
        cand = inst.rows_np[e]
        score = inst.costs[cand] / inst.gain[cand]
        ok = cand[score <= score.min() * (1 + delta)]
        inst.add_set(int(ok[rng.integers(ok.size)]))
    return inst.solution_cost


def _construct(inst, origin, k, rng, regret, delta):
    """One constructive run. Returns (constructive cost, randomised?). k = 0 -> deterministic CH2 / CH3."""
    if origin == "CH1":
        return ch1_random(inst, seed=int(rng.integers(2**31 - 1))), True
    if origin == "CH2":
        return (ch2_greedy(inst), False) if k == 0 else (ch2_randomized(inst, rng, delta), True)
    return (ch3_regret(inst, regret=regret), False) if k == 0 else (ch3_randomized(inst, regret, rng, delta), True)


# ----------------------------------------------------------------------------
# Initial population
# ----------------------------------------------------------------------------
def build_initial_population(inst, pop_size=POP_SIZE, shares=POP_SHARES, delta=RCL_DELTA,
                             seed=GA_SEED, max_dup_tries=MAX_DUP_TRIES, regret=None):
    """Initial GA population made with CH1, CH2 and CH3 (element-to-set encoding).

    Returns {"chromosomes": int array (pop_size, m), "info": DataFrame with one row per individual}.
    `Constructive_Cost` is the cost of the heuristic's binary solution, `Decoded_Cost` the cost of
    the decoded chromosome (<= constructive cost: unused sets disappear)."""
    if regret is None:
        regret = compute_regret(inst)
    n1 = max(1, round(shares["CH1"] * pop_size))
    n2 = max(1, round(shares["CH2"] * pop_size))
    n3 = pop_size - n1 - n2
    if n3 < 1:
        raise ValueError("pop_size too small for the requested shares")
    plan = [("CH1", k) for k in range(n1)] + [("CH2", k) for k in range(n2)] + [("CH3", k) for k in range(n3)]

    rng = np.random.default_rng(seed)
    chromosomes, records, seen = [], [], set()
    for origin, k in plan:
        for _ in range(max_dup_tries):
            t0 = time.perf_counter()
            ch_cost, randomized = _construct(inst, origin, k, rng, regret, delta)
            chosen_ch = inst.chosen.copy()
            chrom = encode_solution(inst, chosen_ch)
            elapsed = time.perf_counter() - t0
            if chrom.tobytes() not in seen or not randomized:
                break
        chosen_dec, dec_cost = decode_chromosome(inst, chrom)
        records.append({"Origin": origin, "Randomized": randomized,
                        "Constructive_Cost": float(ch_cost), "Constructive_Sets": int(chosen_ch.sum()),
                        "Decoded_Cost": dec_cost, "Decoded_Sets": int(chosen_dec.sum()),
                        "Redundant_Sets": count_redundant_sets(inst, chosen_dec),
                        "Duplicate": chrom.tobytes() in seen, "Time_s": elapsed})
        seen.add(chrom.tobytes())
        chromosomes.append(chrom)
    return {"chromosomes": np.vstack(chromosomes), "info": pd.DataFrame(records)}


def validate_population(inst, pop):
    """Every chromosome: genes in rows[e], decoded solution is a cover, stored cost is right,
    and the decoded cost is not higher than the constructive cost."""
    info = pop["info"]
    assert pop["chromosomes"].shape == (len(info), inst.m)
    for chrom, row in zip(pop["chromosomes"], info.itertuples()):
        assert is_valid_chromosome(inst, chrom), f"{inst.name}: gene outside rows[e]"
        chosen, cost = decode_chromosome(inst, chrom)
        assert covers_all_elements(inst, chosen), f"{inst.name}: decoded solution is not a cover"
        assert abs(cost - row.Decoded_Cost) < 1e-6, f"{inst.name}: stored cost differs"
        assert cost <= row.Constructive_Cost + 1e-6, f"{inst.name}: decoding increased the cost"

### 8.4 Build and validate the initial population for one instance

Set `GA_LABEL` to the instance you want to work on. `POP_SIZE`, `POP_SHARES` (CH1 / CH2 / CH3) and `RCL_DELTA` are defined in the cell above. The validation checks, for every individual, that all genes are admissible, that the decoded solution is a cover, and that the cost is consistent.

In [33]:
GA_LABEL = "4.2"                       # instance used by the GA

if GA_LABEL in instances:
    inst = instances[GA_LABEL]
    population = build_initial_population(inst)
    validate_population(inst, population)
    chromosomes, pop_info = population["chromosomes"], population["info"]
    print(f"Instance {GA_LABEL}: m = {inst.m} elements, n = {inst.n} sets, BKS = {BKS[GA_LABEL]}")
    print(f"OK: {len(pop_info)} chromosomes {chromosomes.shape} (rows = individuals, columns = elements, "
          f"value = covering set) are admissible and decode to feasible covers.")
else:
    population = chromosomes = pop_info = None
    print(f"Instance {GA_LABEL!r} not found. Available: {list(instances)}")

Instance 4.2: m = 200 elements, n = 1000 sets, BKS = 512
OK: 30 chromosomes (30, 200) (rows = individuals, columns = elements, value = covering set) are admissible and decode to feasible covers.


In [34]:
# ---- The population ---------------------------------------------------------------------------------
if population is not None:
    bks = float(BKS[GA_LABEL])
    pop_info["Pct_Dev"] = 100 * (pop_info["Decoded_Cost"] - bks) / bks     # decoded cost vs BKS
    display(pop_info.round(3))

    print("By origin:")
    display(pop_info.groupby(["Origin", "Randomized"]).agg(
        Individuals=("Pct_Dev", "size"), Mean_Pct_Dev=("Pct_Dev", "mean"), Best_Pct_Dev=("Pct_Dev", "min"),
        Worst_Pct_Dev=("Pct_Dev", "max"), Mean_Time_s=("Time_s", "mean")).round(3))
    print(f"Distinct chromosomes: {int((~pop_info['Duplicate']).sum())} of {len(pop_info)}")
    print(f"Cost saved by decoding (unused sets drop out): "
          f"{100 * (1 - pop_info['Decoded_Cost'].sum() / pop_info['Constructive_Cost'].sum()):.1f} %")
    for o in ("CH1", "CH2", "CH3"):
        i = pop_info.index[pop_info["Origin"] == o][0]
        print(f"{o} individual {i}: first 15 genes = {chromosomes[i][:15].tolist()}")

,Origin,Randomized,Constructive_Cost,Constructive_Sets,Decoded_Cost,Decoded_Sets,Redundant_Sets,Duplicate,Time_s,Pct_Dev
0,CH1,True,4132.0,83,2959.0,63,0,False,0.006,477.930
1,CH1,True,4448.0,88,3340.0,69,1,False,0.004,552.344
2,CH1,True,4046.0,80,3228.0,64,0,False,0.003,530.469
3,CH1,True,4162.0,88,3138.0,67,0,False,0.002,512.891
4,CH1,True,3706.0,81,2719.0,63,0,False,0.002,431.055
5,CH1,True,4404.0,88,3285.0,67,1,False,0.004,541.602
6,CH1,True,4354.0,91,3114.0,64,1,False,0.002,508.203
7,CH1,True,4198.0,83,3007.0,63,0,False,0.002,487.305
8,CH1,True,4427.0,82,3223.0,64,2,False,0.002,529.492
9,CH1,True,3999.0,84,2789.0,63,0,False,0.001,444.727


By origin:


Individuals  Mean_Pct_Dev  Best_Pct_Dev  Worst_Pct_Dev  \
Origin Randomized                                                           
CH1    True                 15       514.102       431.055        566.602   
CH2    False                 1         3.320         3.320          3.320   
       True                  7         4.911         3.516          7.812   
CH3    False                 1        28.906        28.906         28.906   
       True                  6        23.633        13.672         31.445   

                   Mean_Time_s  
Origin Randomized               
CH1    True              0.002  
CH2    False             0.007  
       True              0.003  
CH3    False             0.001  
       True              0.002

Distinct chromosomes: 30 of 30
Cost saved by decoding (unused sets drop out): 22.8 %
CH1 individual 0: first 15 genes = [934, 706, 433, 316, 885, 369, 238, 366, 883, 664, 773, 347, 404, 962, 124]
CH2 individual 15: first 15 genes = [17, 16, 293, 16, 16, 176, 17, 111, 293, 102, 68, 18, 162, 47, 112]
CH3 individual 23: first 15 genes = [17, 16, 433, 16, 16, 233, 17, 111, 20, 342, 68, 18, 233, 47, 433]
